# Práctica 7 · Que alguien más lo pueda usar

Tienes un sistema que funciona. Busca bien, responde con fundamento, se defiende de documentos
manipulados y sabe cuándo callarse. Y hasta ahora solo lo puedes usar tú, escribiendo en celdas
de un cuaderno.

Hoy le ponemos una interfaz. No es un adorno: es la diferencia entre un ejercicio y algo que
puedes poner delante de un cliente, de un jefe o de un comité.

Y hay una razón menos obvia para hacerlo. Muchas decisiones que tomamos en las prácticas
anteriores solo se vuelven visibles cuando alguien usa el sistema de verdad. Que la respuesta
tarde diez segundos, que no se sepa de dónde salió un dato, que el sistema conteste con seguridad
algo que no debería. Nada de eso se nota ejecutando celdas.

Las celdas se ejecutan en orden, una por una, con Shift + Enter.

## 1. Instalar las librerías

Gradio arma interfaces web desde Python, sin escribir HTML ni JavaScript. Es la herramienta
habitual para prototipos de este tipo, y trae un componente de chat ya hecho.

Un aviso por si buscas ejemplos en internet: esta librería cambia de interfaz seguido, y mucho
código que encontrarás usa parámetros que ya no existen. Si algo falla con un error de argumento
inesperado, revisa la firma con `inspect.signature` antes de pelearte con el ejemplo.

In [1]:
%pip install --quiet gradio pymupdf rank_bm25

print("Librerías listas.")


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /Users/fgrodriguez/ESAN_GlobalWeek2026/09_Notebooks_RAG/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Librerías listas.


## 2. Comprobar que Ollama responde

Ollama es el programa que ejecuta los modelos de lenguaje dentro de tu computadora. Tiene que
estar encendido para que este cuaderno funcione, así que lo primero es confirmarlo.

Si algo falla, la salida de la celda te dice qué hacer según tu sistema operativo.

In [2]:
import platform
import sys

import requests

OLLAMA_URL = "http://localhost:11434"

print(f"Sistema: {platform.system()} {platform.machine()}")
print(f"Python:  {sys.version.split()[0]}\n")

try:
    respuesta = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    respuesta.raise_for_status()
    modelos = sorted(m["name"] for m in respuesta.json()["models"])
    print(f"Ollama responde. Tienes {len(modelos)} modelos descargados:\n")
    for m in modelos:
        print(f"  - {m}")
except Exception as e:
    print(f"Ollama no responde en {OLLAMA_URL}")
    print(f"Detalle: {type(e).__name__}\n")
    if platform.system() == "Darwin":
        print("En Mac: abre la aplicación Ollama desde la carpeta Aplicaciones.")
        print("Debe aparecer su ícono en la barra de menús, arriba a la derecha.")
    elif platform.system() == "Windows":
        print("En Windows: busca Ollama en el menú Inicio y ábrelo.")
        print("Debe aparecer su ícono junto al reloj, abajo a la derecha.")
    else:
        print("Ejecuta 'ollama serve' en una terminal.")

Sistema: Darwin arm64
Python:  3.12.13

Ollama responde. Tienes 14 modelos descargados:

  - embeddinggemma:300m
  - gemma3:1b
  - gemma3:4b
  - gemma4:12b-mlx
  - gemma4:26b
  - gemma4:e4b
  - granite4.1:3b
  - granite4.1:8b
  - mxbai-embed-large:latest
  - nemotron-mini:4b
  - nomic-embed-text:latest
  - qwen3-4b-cs-ft:latest
  - qwen3:4b
  - shieldgemma:2b


## 3. Elegir el modelo según tu equipo

El modelo va a correr en tu máquina, así que la memoria que tengas importa. Un modelo grande
en un equipo chico no se rompe: simplemente tarda muchísimo y el sistema se pone lento.

Abajo hay tres opciones. Deja activa una sola, la que corresponda a tu computadora, y comenta
las demás poniéndoles un signo de gato al inicio de la línea. Si no sabes cuánta memoria
tienes, quédate con la opción A, que funciona en cualquier equipo.

El modelo de embeddings no se elige por equipo: es ligero y va igual en todos. Sí conviene saber
de dónde salió esa elección, y la respuesta es que está medida con documentos en español; en la
práctica 3 vas a reproducir la medición y a ver a los tres candidatos compitiendo.

In [3]:
# ---- Opción A: equipos de 8 GB de memoria o menos (descarga 3.3 GB) ---------
MODELO_LLM = "gemma3:4b"

# ---- Opción B: equipos de 16 GB de memoria (descarga 10 GB) ----------------
# MODELO_LLM = "gemma4:12b"        # Windows y Linux
# MODELO_LLM = "gemma4:12b-mlx"    # Mac con chip Apple (M1 en adelante), va más rápido

# ---- Opción C: equipos de 32 GB de memoria o más (descarga 17 GB) ----------
# MODELO_LLM = "gemma4:26b"        # Windows y Linux
# MODELO_LLM = "gemma4:26b-mlx"    # Mac con chip Apple

# El modelo de embeddings es ligero y es el mismo para todos. La elección está
# medida, no copiada de un tutorial: lo comprobamos en la práctica 3. Se eligió
# éste porque es el único de los tres que encuentra un pasaje en inglés cuando la
# pregunta va en español, algo que hace falta en cuanto el corpus mezcla idiomas.
MODELO_EMBEDDINGS = "embeddinggemma:300m"

print(f"Modelo de lenguaje:   {MODELO_LLM}")
print(f"Modelo de embeddings: {MODELO_EMBEDDINGS}")
print("\nSi alguno no aparece en la lista de la celda anterior, descárgalo con:")
print(f"   ollama pull {MODELO_LLM}")
print(f"   ollama pull {MODELO_EMBEDDINGS}")

Modelo de lenguaje:   gemma3:4b
Modelo de embeddings: embeddinggemma:300m

Si alguno no aparece en la lista de la celda anterior, descárgalo con:
   ollama pull gemma3:4b
   ollama pull embeddinggemma:300m


## 4. Montar el sistema completo

En una sola celda juntamos todo lo de las prácticas anteriores: parseo separando tablas, índice
vectorial, filtro contra documentos manipulados y la cadena de respuesta.

No hay nada nuevo aquí. Si algo no te suena, está explicado paso a paso en su práctica.

In [4]:
import re
import warnings
warnings.filterwarnings("ignore", message=".*langchain-community.*")

from pathlib import Path

import pymupdf
from langchain_community.vectorstores import LanceDB
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Traemos la defensa de la práctica 5. Aquí ya no es un ejercicio: el filtro corre
# sobre el corpus real que va a alimentar la interfaz.
PATRONES_DE_INYECCION = [
    r"ignora\w*\s+(todas\s+)?(las\s+)?instrucciones",
    r"olvida\w*\s+(todo\s+)?lo\s+anterior",
    r"nueva\s+directiva",
    r"no\s+menciones\s+est[ea]",
    r"ignore\s+(all\s+)?previous\s+instructions",
]


def parece_manipulado(texto):
    return any(re.search(p, texto, re.I) for p in PATRONES_DE_INYECCION)


def tabla_a_frases(filas):
    encabezados = filas[0]
    return ["; ".join(f"{e}: {v}" for e, v in zip(encabezados, fila) if v)
            for fila in filas[1:]]


# Todo lo de las prácticas 2 y 5 junto: separar tablas de texto, guardar el origen de
# cada fragmento y descartar los que parezcan manipulados. La página se guarda con
# +1 porque pymupdf las numera desde cero y las personas desde uno.
def cargar_corpus(carpeta, saltar=()):
    divisor = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    fragmentos, descartados = [], 0

    for ruta in sorted(Path(carpeta).glob("*.pdf")):
        if ruta.name in saltar:
            continue
        doc = pymupdf.open(ruta)
        for pagina in doc:
            tablas = pagina.find_tables().tables
            for tabla in tablas:
                for frase in tabla_a_frases(tabla.extract()):
                    if parece_manipulado(frase):
                        descartados += 1
                        continue
                    fragmentos.append(Document(
                        page_content=frase,
                        metadata={"fuente": ruta.name, "pagina": pagina.number + 1}))

            recortes = [pagina.get_text(clip=t.bbox) for t in tablas]
            lineas = [l.strip() for l in pagina.get_text().split("\n")
                      if l.strip() and not any(l.strip() in r for r in recortes)]
            for trozo in divisor.split_text(" ".join(lineas)):
                if parece_manipulado(trozo):
                    descartados += 1
                    continue
                fragmentos.append(Document(
                    page_content=trozo,
                    metadata={"fuente": ruta.name, "pagina": pagina.number + 1}))

    return fragmentos, descartados


# Dejamos fuera el libro en inglés: aquí queremos un asistente de TiendaSol.
corpus, descartados = cargar_corpus("documentos", saltar=("Alice_in_Wonderland.pdf",))

embeddings = OllamaEmbeddings(model=MODELO_EMBEDDINGS)
almacen = LanceDB.from_documents(corpus, embeddings, uri="lancedb_p7",
                                 table_name="asistente", mode="overwrite")
# El modelo que redacta la respuesta. temperature=0 hace que, ante la misma pregunta,
# conteste siempre lo mismo: justo lo que se quiere en atención al cliente.
#
# Aquí es donde una organización pondría la llamada a un servicio comercial si
# decidiera no usar un modelo local. Este curso corre en local a propósito.
llm = ChatOllama(model=MODELO_LLM, temperature=0)

print(f"Fragmentos indexados:            {len(corpus)}")
print(f"Fragmentos descartados por sucios: {descartados}")

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


Fragmentos indexados:            27
Fragmentos descartados por sucios: 2


## 5. La función que responde

Una diferencia con las prácticas anteriores: además de la respuesta, esta función devuelve de
dónde salió. La interfaz lo necesita para poder mostrar las fuentes.

In [5]:
# La plantilla que va a producción. Le dice quién es, de dónde puede sacar los datos,
# qué hacer cuando no sabe y de qué largo responder. Cada una de esas líneas está por
# un problema concreto visto en las prácticas anteriores.
PLANTILLA = ChatPromptTemplate.from_template(
    """Eres el asistente del centro de ayuda de TiendaSol.

Responde la pregunta del cliente usando únicamente la información del contexto.
Si el contexto no contiene la respuesta, dilo con claridad y ofrece pasar con un
asesor. No inventes datos, precios ni plazos.
Responde en español, en dos o tres oraciones.

Pregunta del cliente: {pregunta}

Contexto:
{contexto}"""
)


# Devuelve dos cosas: la respuesta y los documentos en que se apoyó. Lo segundo es lo
# que permite mostrar las fuentes al cliente, y es la diferencia entre un asistente
# que se puede auditar y uno al que hay que creerle.
def responder(pregunta, k=4):
    """Devuelve la respuesta y los fragmentos en que se apoyó."""
    documentos = almacen.similarity_search(pregunta, k=k)
    contexto = "\n\n".join(f"[{d.metadata['fuente']} p.{d.metadata['pagina']}] "
                            f"{d.page_content}" for d in documentos)
    respuesta = (PLANTILLA | llm | StrOutputParser()).invoke(
        {"pregunta": pregunta, "contexto": contexto}).strip()
    return respuesta, documentos


respuesta, fuentes = responder("¿Cuánto cuesta el envío a Lima?")
print(respuesta)
print("\nFuentes:")
for d in fuentes[:3]:
    print(f"  {d.metadata['fuente']} p.{d.metadata['pagina']}")

El costo del envío a Lima es de S/ 9.90. Recuerda que el envío es gratuito a partir de S/ 99 de compra. Si necesitas más detalles sobre las opciones de envío, te pasamos con un asesor para ayudarte.

Fuentes:
  tiendasol_politicas.pdf p.1
  tiendasol_politicas.pdf p.1
  tiendasol_politicas.pdf p.1


## 6. La interfaz mínima

Con eso ya se puede armar un chat. Gradio necesita una función que reciba el mensaje y el
historial, y devuelva el texto de la respuesta.

Es deliberadamente simple, para que veas el punto de partida antes de mejorarlo.

In [6]:
# Gradio arma una página web a partir de una función de Python. No hay que escribir
# HTML ni JavaScript: se le dice qué función llamar y con qué controles.
import gradio as gr


def conversar(mensaje, historial):
    respuesta, _ = responder(mensaje)
    return respuesta


# ChatInterface es la versión de una línea: se le pasa la función y arma el chat
# completo. Sirve para una demostración, pero se queda corta en cuanto hacen falta
# fuentes, votos o mensajes de espera. Por eso abajo se rehace con Blocks.
chat_simple = gr.ChatInterface(
    fn=conversar,
    title="Centro de Ayuda TiendaSol",
    description="Preguntas sobre envíos, devoluciones y pedidos",
    examples=["¿Cuánto cuesta el envío a Lima?",
              "¿Cuántos días tengo para devolver algo?",
              "¿Qué significa el estado EST-07?"],
)

print("Interfaz definida. La lanzamos en la sección 8, ya mejorada.")

/Users/fgrodriguez/ESAN_GlobalWeek2026/09_Notebooks_RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Interfaz definida. La lanzamos en la sección 8, ya mejorada.


## 7. Lo que le falta a esa interfaz

Antes de lanzarla, vale la pena mirar qué le falta, porque es lo que separa una demostración de
algo utilizable.

No dice de dónde salió la respuesta. El cliente lee un párrafo con tono seguro y no tiene forma
de verificarlo. Es lo primero que pregunta cualquier área de cumplimiento.

No avisa que está trabajando. Con un modelo local, una respuesta tarda varios segundos, y sin
señal alguna el usuario cree que se colgó.

No permite decir que la respuesta estuvo mal. Sin ese canal, nunca sabrás qué preguntas contesta
mal tu sistema, y son exactamente las que necesitas para mejorarlo.

No filtra lo que no debe contestar. Lo que hicimos en la práctica 5 se quedó en el cuaderno.

Vamos a agregarle las cuatro cosas.

In [7]:
import json
from datetime import datetime

# Un archivo .jsonl guarda un registro por línea. Es el formato más simple que permite
# ir agregando sin releer lo anterior, y se puede abrir con cualquier herramienta.
REGISTRO = Path("retroalimentacion.jsonl")

# El filtro de la práctica 5, ahora en el camino real de la consulta. Se revisa ANTES
# de buscar: si la pregunta no corresponde, no hay que gastar en recuperar ni generar.
REGLAS_DEL_NEGOCIO = [
    r"\b(datos?|informaci[oó]n)\b.{0,25}\b(de|del)\b.{0,15}\bcliente\b",
    r"\b(tarjeta|n[uú]mero de tarjeta|cvv|contrase[nñ]a)\b",
    r"\b(lista|listado|exporta|dame todos)\b.{0,20}\b(clientes?|correos?|usuarios?)\b",
]


def fuera_de_alcance(pregunta):
    return any(re.search(p, pregunta, re.I) for p in REGLAS_DEL_NEGOCIO)


# Esta es la función que ve el cliente. Tres cosas en orden: filtrar, responder y citar.
def conversar_completo(mensaje, historial):
    """Responde citando fuentes, o rechaza si la consulta no corresponde."""
    if fuera_de_alcance(mensaje):
        return ("No puedo ayudarte con datos personales o de otros clientes. "
                "Si necesitas información de tu cuenta, te comunico con un asesor.")

    respuesta, documentos = responder(mensaje)

    # Varios fragmentos pueden venir de la misma página. El conjunto (set) evita que la
    # misma fuente aparezca repetida en la lista que se le muestra al cliente.
    vistas, lineas = set(), []
    for d in documentos:
        clave = (d.metadata["fuente"], d.metadata["pagina"])
        if clave not in vistas:
            vistas.add(clave)
            lineas.append(f"- {d.metadata['fuente']}, página {d.metadata['pagina']}")

    return respuesta + "\n\n---\n**Fuentes consultadas**\n" + "\n".join(lineas)


# Los pulgares del chat. Sin este registro no hay forma de saber si el asistente está
# sirviendo: es el dato más barato de recoger y el primero que pide un jefe.
def guardar_voto(datos: gr.LikeData):
    """Guarda si el usuario aprobó o rechazó una respuesta."""
    with REGISTRO.open("a") as f:
        f.write(json.dumps({
            "momento": datetime.now().isoformat(timespec="seconds"),
            "util": datos.liked,
            "respuesta": str(datos.value)[:400],
        }, ensure_ascii=False) + "\n")


print("Funciones de la interfaz listas.")

Funciones de la interfaz listas.


## 8. La interfaz completa

Ahora sí. Al ejecutar esta celda aparece el chat aquí mismo, debajo de la celda, y también queda
disponible en tu navegador en la dirección que imprime.

In [8]:
# Blocks arma la página pieza por pieza y deja controlar cómo se conectan entre sí.
# Todo lo que va indentado aquí adentro es un elemento de la pantalla.
with gr.Blocks(title="Centro de Ayuda TiendaSol") as asistente:
    gr.Markdown("## Centro de Ayuda TiendaSol\n"
                "Consultas sobre envíos, devoluciones, pedidos y códigos de referencia.")

    chat = gr.Chatbot(height=380)
    entrada = gr.Textbox(placeholder="Escribe tu pregunta y presiona Enter",
                         show_label=False, autofocus=True)

    with gr.Row():
        gr.Examples(
            examples=["¿Cuánto cuesta el envío a Lima?",
                      "¿Cuántos días tengo para devolver un producto?",
                      "¿Qué significa el estado EST-07?",
                      "¿Puedo pagar en criptomonedas?"],
            inputs=entrada,
        )

    # yield entrega un resultado y sigue trabajando. El primero pinta el aviso de
    # "consultando" al instante, y el segundo lo reemplaza con la respuesta cuando
    # está lista. Sin eso el cliente mira una pantalla quieta varios segundos.
    def turno(mensaje, historial):
        historial = historial + [{"role": "user", "content": mensaje}]
        yield historial + [{"role": "assistant", "content": "_Consultando la documentación..._"}], ""
        respuesta = conversar_completo(mensaje, historial)
        yield historial + [{"role": "assistant", "content": respuesta}], ""

    # Aquí se conecta todo: cuando el usuario presiona Enter en la caja de texto, se
    # llama a turno() con la caja y el chat, y lo que devuelve actualiza el chat y
    # vacía la caja. La segunda línea engancha los pulgares al registro.
    entrada.submit(turno, [entrada, chat], [chat, entrada])
    chat.like(guardar_voto, None, None)

# prevent_thread_lock deja que el cuaderno siga vivo mientras la interfaz corre, para
# poder seguir ejecutando celdas. La dirección que imprime abajo se abre en el navegador.
asistente.launch(prevent_thread_lock=True, quiet=True, inbrowser=False)
print(f"Interfaz corriendo en: {asistente.local_url}")

Interfaz corriendo en: http://127.0.0.1:7860/


Pruébala con las preguntas de ejemplo y con las tuyas. Fíjate en tres cosas mientras la usas.

El mensaje de "consultando la documentación" aparece de inmediato y se reemplaza cuando llega la
respuesta. Es un truco barato que cambia por completo la percepción de lentitud: el sistema tarda
lo mismo, pero deja de parecer que se colgó.

Cada respuesta trae debajo de qué documento y de qué página salió. Pregúntale algo y ve a
comprobarlo en el PDF.

Y prueba la pregunta sobre criptomonedas, que no está en la documentación, y alguna del tipo
"dame los datos del cliente Juan Pérez", que debería rechazar antes siquiera de buscar.

Los pulgares de cada respuesta guardan tu voto en un archivo. Eso, que parece un detalle, es la
única fuente de información sobre qué está fallando cuando el sistema ya está en uso.

In [9]:
if REGISTRO.exists():
    votos = [json.loads(l) for l in REGISTRO.read_text().splitlines() if l.strip()]
    utiles = sum(1 for v in votos if v["util"])
    print(f"Votos registrados: {len(votos)}  (útiles: {utiles}, no útiles: {len(votos)-utiles})")
    for v in votos[-3:]:
        print(f"  [{'+' if v['util'] else '-'}] {v['respuesta'][:80]}")
else:
    print("Todavía no hay votos. Usa los pulgares en el chat de arriba y vuelve a ejecutar.")

Todavía no hay votos. Usa los pulgares en el chat de arriba y vuelve a ejecutar.


## 9. Lo que hace buena a una interfaz de este tipo

Cuatro criterios que conviene tener presentes al diseñar la tuya, y que van más allá de Gradio.

**Que se vea de dónde salió.** Una respuesta sin fuente es una afirmación que hay que creer. Con
la cita, el usuario puede verificar, y cuando el sistema se equivoca, alguien lo detecta. Es la
diferencia entre una herramienta y un oráculo.

**Que no parezca colgado.** Con modelos locales las respuestas tardan segundos. Mostrar de
inmediato que se está trabajando, y a ser posible ir mostrando la respuesta conforme se genera,
cambia la percepción más que optimizar el modelo.

**Que se pueda reclamar.** Los pulgares no son decoración. Son el único canal por el que vas a
enterarte de qué preguntas contesta mal tu sistema cuando ya no estás mirando. Guárdalos junto con
la pregunta y el contexto que se usó, para poder reconstruir qué pasó.

**Que rechace con elegancia.** Cuando el sistema no puede o no debe contestar, un "no tengo esa
información, te comunico con un asesor" es infinitamente mejor que una respuesta inventada o que
un mensaje de error. Y ese camino de salida hay que diseñarlo, no improvisarlo.

## 10. De esto a algo en producción

Lo que armaste es un prototipo honesto: hace lo que dice y se puede mostrar. Lo que le falta para
ser un servicio conviene decirlo con claridad, porque es la pregunta que va a hacer cualquiera que
lo vea funcionando.

Corre en tu máquina y para un usuario a la vez. Un servicio real necesita atender a varias
personas al mismo tiempo, y ahí aparecen las colas, los tiempos de espera y el dimensionamiento
del servidor.

No tiene sesiones ni memoria de la conversación. Cada pregunta se responde de cero, así que un
"¿y cuánto tarda?" después de preguntar por el envío no funciona.

No tiene control de acceso. Cualquiera que abra la dirección ve todo, y en un sistema con
documentos internos eso no es aceptable.

Y no guarda registro de lo que respondió. Los votos sí se guardan, pero no las respuestas
completas ni el contexto usado, que es lo que hace falta para investigar un reclamo.

Nada de eso cambia lo que aprendiste. El motor que armaste en las siete prácticas es el mismo que
lleva dentro un sistema en producción; lo que se agrega alrededor es infraestructura, y es un
problema conocido con soluciones conocidas.

## 11. Cierre del recorrido

Vale la pena mirar atrás un momento.

Empezaste con un sistema que respondía sobre una novela y no sabía de dónde sacaba nada.
Aprendiste a preparar documentos separando lo que sirve de lo que estorba, a elegir el modelo de
embeddings midiendo en lugar de copiando, a buscar y a decidir cuántos fragmentos traer, a
escribirle instrucciones al modelo y a comprobar si las cumplió. Después le agregaste técnicas
avanzadas y comprobaste que una de ellas no aportaba nada en tu caso. Lo defendiste de un
documento manipulado y descubriste que la defensa recomendada no bastaba. Mediste cómo escala y
qué se rompe de verdad. Y hoy lo pusiste detrás de una interfaz.

Si tuviera que quedarme con una sola cosa de todo el recorrido, no sería ninguna técnica. Sería el
hábito de medir antes de decidir. Casi todas las sorpresas de estas prácticas fueron eso: una
creencia razonable que la medición desmintió. El modelo de embeddings que todos usan resultó ser
el peor para nuestro caso. La búsqueda híbrida, que es la recomendación estrella, no aportó nada.
La defensa contra inyección que recomienda la literatura falló con el modelo pequeño. El traslape
entre fragmentos infla menos de lo que se calcula a mano.

Ninguna de esas cosas se sabe leyendo. Se saben probando, con tus documentos, tus preguntas y tu
equipo. Eso es lo que te llevas.